# Case-loss training ablation — CMPE 586
Same initialization, row views, retained word plans and numerical recipe as the completed augmented model; omit only case loss. No budget redistribution.

Plan: `configs/ablation-case-loss.json`. Calibration, formal training and archive creation are separate and disabled by default. The calibration's updated weights are discarded. No held-out test evaluation or Drive deletion is performed here.

In [ ]:
import json
import os
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path

if sys.version_info[:2] != (3, 12) or version("torch") != "2.11.0+cu128":
    raise RuntimeError("Select the matched Colab 2026.07 GPU runtime.")
PROJECT=Path("/content/cmpe586-turkish-punctuation")
if not PROJECT.exists():
    subprocess.run(["git","clone","https://github.com/kckagancan/cmpe586-turkish-punctuation.git",str(PROJECT)],check=True)
else:
    if subprocess.check_output(["git","status","--porcelain"],cwd=PROJECT,text=True).strip():
        raise RuntimeError("Review checkout changes before updating.")
    subprocess.run(["git","pull","--ff-only","origin","main"],cwd=PROJECT,check=True)
subprocess.run([sys.executable,"-m","pip","install","-e",str(PROJECT),"-c",str(PROJECT/"requirements/colab.constraints.txt")],check=True)
sys.path.insert(0,str(PROJECT/"src"))
os.environ["TOKENIZERS_PARALLELISM"]="false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"]="1"
print("Source commit:",subprocess.check_output(["git","rev-parse","HEAD"],cwd=PROJECT,text=True).strip())


## Project storage and fixed experiment
Reuse the existing project mount and persistent shared initialization. Select the same T4 numerical environment as the two completed training arms.

In [ ]:
import torch
from turkish_punctuation.ablation import load_plan, training_command, verify_ready

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required; CPU training is not substituted.")
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive",timeout_ms=600_000)
STORAGE=Path("/content/drive/MyDrive/cmpe586-turkish-punctuation")
if not STORAGE.is_dir():
    raise RuntimeError("The existing project folder is not mounted.")
PLAN,TRAINING_CONFIG=load_plan()
TRANSFER=json.loads((PROJECT/"configs/ablation-case-loss-inputs.json").read_text())
VIEWS=STORAGE/PLAN["storage_views"]
CALIBRATION_OUTPUT=STORAGE/PLAN["storage_calibration"]
TRAIN_OUTPUT=STORAGE/PLAN["storage_training"]
READINESS=STORAGE/PLAN["storage_preparation"]/"readiness.json"
print("GPU:",torch.cuda.get_device_name(0))
print("Omitted family:",PLAN["without_family"])
print("Formal output:",TRAIN_OUTPUT)


## Restore prepared inputs — zero optimizer updates
The ZIP was prepared and re-tokenized locally. Verify and restore it; never regenerate these views on Drive. Reuse only the train/validation clean-cache entries and pinned raw splits. The runner rechecks encodings before optimizer creation.

In [ ]:
sys.path.insert(0,str(PROJECT/"scripts"))
from prepare_training import copy_verified_tree
from turkish_punctuation.training_io import file_hashes, sha256

clean_run=json.loads((STORAGE/PLAN["reference_runs"]["clean"]["storage_directory"]/"run.json").read_text())
clean_inputs=clean_run["specification"]["inputs"]
clean_cache=STORAGE/"inputs/clean-v1/prepared"
local_clean=PROJECT/TRAINING_CONFIG["data"]["prepared_directory"]
if sha256(clean_cache/"manifest.json")!=clean_inputs["prepared_manifest_sha256"]:
    raise RuntimeError("The persistent clean manifest differs from the original pair.")
local_clean.mkdir(parents=True,exist_ok=True)
local_manifest=local_clean/"manifest.json"
if local_manifest.exists() and sha256(local_manifest)!=clean_inputs["prepared_manifest_sha256"]:
    raise RuntimeError("Local clean manifest differs; no overwrite performed.")
if not local_manifest.exists():
    local_manifest.write_bytes((clean_cache/"manifest.json").read_bytes())
for split in ("train","validation"):
    cached=clean_cache/"dataset"/split
    if file_hashes(cached)!=clean_inputs[split]["prepared_files"]:
        raise RuntimeError(f"Persistent {split} bytes differ from the completed pair.")
    copy_verified_tree(cached,local_clean/"dataset"/split)
subprocess.run([sys.executable,"scripts/download_data.py","--splits","train","validation"],cwd=PROJECT,check=True)
if VIEWS.exists():
    subprocess.run([sys.executable,"scripts/prepare_ablation.py","audit","--storage-root",str(STORAGE)],cwd=PROJECT,check=True)
else:
    subprocess.run([sys.executable,"scripts/prepare_ablation.py","restore","--storage-root",str(STORAGE),
                    "--archive",str(STORAGE/PLAN["storage_input_zip"]),"--sha256",TRANSFER["sha256"]],cwd=PROJECT,check=True)
prepared=json.loads((VIEWS/"manifest.json").read_text())
if prepared["identity"]!=TRANSFER["prepared_identity"]:
    raise RuntimeError("Restored views differ from the audited local preparation.")
for epoch,expected in TRANSFER["epoch_audit"].items():
    if prepared["epochs"][epoch]["audit"]["encoded_content_sha256"]!=expected["encoded_content_sha256"]:
        raise RuntimeError(f"Epoch {epoch}: encoded content differs.")
print("ABLATION_INPUTS_READY — views restored; zero optimizer updates.")


In [ ]:
def stream_command(command):
    import signal
    with subprocess.Popen(command,cwd=PROJECT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            assert process.stdout is not None
            for line in process.stdout:
                print(line,end="",flush=True)
            status=process.wait()
        except KeyboardInterrupt:
            if process.poll() is None:
                process.send_signal(signal.SIGINT)
            raise
        if status:
            raise subprocess.CalledProcessError(status,command)


## Short GPU calibration — disabled
A fresh 30-update timing/memory check includes the longest batch from the longest epoch view. These are technical calibration updates, not a formal ablation result. Updated weights are discarded; the formal model starts from the shared initialization again.

In [ ]:
RUN_CALIBRATION=False
if RUN_CALIBRATION:
    if (CALIBRATION_OUTPUT/"calibration.json").exists():
        print("Existing calibration retained; the readiness cell verifies it.")
    else:
        stream_command(training_command(STORAGE,mode="calibrate"))
else:
    print("Calibration disabled.")


## Readiness gate — zero additional updates
Verify row masks, retained word plans, calibration events/losses, source/configuration hashes, reference runs, initial weights and the current GPU environment. Formal training must still be unstarted.

In [ ]:
if (CALIBRATION_OUTPUT/"calibration.json").exists() and not TRAIN_OUTPUT.exists():
    stream_command([sys.executable,"scripts/prepare_ablation.py","ready","--storage-root",str(STORAGE),"--output",str(READINESS)])
else:
    print("Readiness not executed: calibration is pending or a formal output already exists.")


## Formal ablation training — disabled
Three epochs / 10,968 optimizer updates, batch 8 × accumulation 4, LR 2e-5, FP32/eager. Select by clean validation macro-F1; tie goes to the earlier epoch. Retain all epoch exports and two fixed optimizer slots.

Set `RUN_FULL_TRAINING=True` only when explicitly starting this experiment. For an interrupted identical run, also set `RESUME=True`. Completed runs are never overwritten.

In [ ]:
RUN_FULL_TRAINING=False
RESUME=False
if RUN_FULL_TRAINING:
    readiness=verify_ready(STORAGE,allow_started=RESUME)
    stream_command(training_command(STORAGE,mode="train",resume=RESUME))
else:
    print("FORMAL ABLATION NOT STARTED — training switch is False.")


## Persistent status
This reads saved metadata only. A passed calibration is separate from completed formal training.

In [ ]:
if (TRAIN_OUTPUT/"summary.json").exists():
    summary=json.loads((TRAIN_OUTPUT/"summary.json").read_text())
    print("ABLATION TRAINING COMPLETE")
    print("Best epoch:",summary["best_epoch"],"Clean macro-F1:",summary["best_macro_f1"])
    print("Formal updates:",summary["global_step"])
else:
    print("FORMAL ABLATION NOT STARTED" if not TRAIN_OUTPUT.exists() else "Formal run exists; inspect progress before resuming.")
if (CALIBRATION_OUTPUT/"calibration.json").exists():
    report=json.loads((CALIBRATION_OUTPUT/"calibration.json").read_text())
    print("Separate calibration:",report["optimizer_updates"],"updates; weights retained:",report["weights_retained"])
print("Formal training switch:",globals().get("RUN_FULL_TRAINING",False))


## Completed-run archive — disabled
After formal completion, build the archive, download it manually, then verify local models/results before scoped Drive cleanup. No deletion is implemented here. The archive includes shared initialization and all candidate epoch models/results; optimizer resume state is omitted for a successfully completed run.

In [ ]:
RUN_ARCHIVE = False
if RUN_ARCHIVE:
    import hashlib
    import json
    import shutil
    import subprocess
    import sys
    from pathlib import Path

    RUN_FULL_TRAINING = False
    RUN_CALIBRATION = False
    archive_project = Path('/content/cmpe586-turkish-punctuation')
    archive_storage = Path('/content/drive/MyDrive/cmpe586-turkish-punctuation')
    archive_run = archive_storage / 'training-v1/train-augmented-without-case-loss-seed-586-v1'
    archive_initialization = archive_storage / 'initializations/training-v1/seed-586'
    archive_run_info = json.loads((archive_run / 'run.json').read_text())
    archive_summary = json.loads((archive_run / 'summary.json').read_text())
    archive_progress = json.loads((archive_run / 'progress.json').read_text())
    if archive_progress['status'] != 'complete' or archive_summary['identity'] != archive_run_info['identity']:
        raise RuntimeError('Archive only the verified completed run.')
    archive_name = f"ablation-without-case-loss-seed586-models-{archive_run_info['identity'][:12]}.zip"
    archive_vm = Path('/content/cmpe586-archives') / archive_name
    archive_vm_receipt = archive_vm.with_suffix('.receipt.json')
    archive_cli = [sys.executable, str(archive_project / 'scripts/archive_training.py')]
    if archive_vm.exists() or archive_vm_receipt.exists():
        if not archive_vm.is_file() or not archive_vm_receipt.is_file():
            raise RuntimeError('Incomplete existing archive transfer; inspect it before retrying.')
        archive_receipt = json.loads(archive_vm_receipt.read_text())
        if archive_receipt['run_identity'] != archive_run_info['identity']:
            raise RuntimeError('Existing archive belongs to another run.')
    else:
        print('Validating completed model/result hashes and building the VM archive...', flush=True)
        archive_receipt = json.loads(subprocess.check_output([
            *archive_cli, '--run-directory', str(archive_run),
            '--initialization-directory', str(archive_initialization), '--output', str(archive_vm)
        ], text=True))
        archive_vm_receipt.write_text(json.dumps(archive_receipt, indent=2) + '\n')
    subprocess.run([
        *archive_cli, '--verify', str(archive_vm), '--expected-sha256', archive_receipt['sha256']
    ], check=True)
    archive_drive = archive_storage / 'archives' / archive_name
    archive_drive.parent.mkdir(parents=True, exist_ok=True)
    if not archive_drive.exists():
        print('Copying the verified ZIP to the project Drive archives folder...', flush=True)
        shutil.copyfile(archive_vm, archive_drive)
    with archive_drive.open('rb') as archive_stream:
        archive_drive_hash = hashlib.file_digest(archive_stream, 'sha256').hexdigest()
    if archive_drive_hash != archive_receipt['sha256'] or archive_drive.stat().st_size != archive_receipt['bytes']:
        raise RuntimeError('Drive ZIP readback differs; do not download or clean sources yet.')
    archive_receipt.update({
        'drive_archive': str(archive_drive), 'source_archive_contents_verified': True,
        'drive_copy_hash_verified': True, 'local_download_verified': False,
        'source_files_deleted': False,
    })
    archive_drive.with_suffix('.receipt.json').write_text(json.dumps(archive_receipt, indent=2) + '\n')
    print('ARCHIVE READY FOR MANUAL DOWNLOAD')
    print(json.dumps(archive_receipt, indent=2))
else:
    print('Archive disabled. Enable only after completed-run verification.')
